In [1]:
from typing import List, Dict
from dataclasses import dataclass

from langchain_community.chat_models import ChatOllama
from langchain_core.messages import HumanMessage


C:\Users\KRISHNENDU\anaconda3\envs\etl-pipeline\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
llm = ChatOllama(model="llama3", temperature=0)


C:\Users\KRISHNENDU\AppData\Local\Temp\ipykernel_7956\1566199341.py:1: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  llm = ChatOllama(model="llama3", temperature=0)


Define a test case schema

In [3]:
@dataclass
class EvalCase:
    input: str
    expected: str
    category: str


Create a golden dataset

In [4]:
eval_dataset: List[EvalCase] = [
    EvalCase(
        input="What is LangChain?",
        expected="framework",
        category="knowledge"
    ),
    EvalCase(
        input="What is 25 * 4?",
        expected="100",
        category="math"
    ),
    EvalCase(
        input="Who is the president of Mars?",
        expected="I don't know",
        category="hallucination"
    )
]


System under test (SUT)

In [5]:
def system_under_test(prompt: str) -> str:
    response = llm.invoke(prompt)
    return response.content


LLM-as-a-Judge (key idea)

In [6]:
def llm_judge(question: str, answer: str, expected: str) -> bool:
    judge_prompt = f"""
You are evaluating an AI answer.

Question:
{question}

AI Answer:
{answer}

Expected concept:
{expected}

Does the answer satisfy the expected concept?
Respond only with YES or NO.
"""
    verdict = llm.invoke(judge_prompt).content.strip()
    return verdict.upper().startswith("YES")


Run evaluation

In [ ]:
results = []

for case in eval_dataset:
    answer = system_under_test(case.input)
    passed = llm_judge(case.input, answer, case.expected)

    results.append({
        "input": case.input,
        "answer": answer,
        "expected": case.expected,
        "category": case.category,
        "passed": passed
    })


Report results

In [ ]:
for r in results:
    status = "✅ PASS" if r["passed"] else "❌ FAIL"
    print(f"{status} | {r['category']} | {r['input']}")


Compute metrics

In [ ]:
accuracy = sum(r["passed"] for r in results) / len(results)
print(f"Overall accuracy: {accuracy:.2f}")


Hallucination detector (simple & effective)

In [ ]:
def hallucination_check(answer: str) -> bool:
    bad_phrases = [
        "as far as I know",
        "it is believed",
        "likely",
        "probably"
    ]
    return any(p in answer.lower() for p in bad_phrases)


In [ ]:
for r in results:
    if r["category"] == "hallucination":
        print("Hallucination detected:", hallucination_check(r["answer"]))


Regression testing (CRITICAL)

In [ ]:
baseline_accuracy = 0.66  # from last release

if accuracy < baseline_accuracy:
    print("🚨 REGRESSION DETECTED")
else:
    print("✅ No regression")
